In [ ]:
import os
import sys
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

In [ ]:
from qasa_rag.embedder import Embedder
from qasa_rag.retrieval import QASARetriever, AnswerAgent

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "musique"  # "musique" or "2wiki"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

QUERY_ENTITY_CACHE_PATH = Path(f"cache/query_entities-{DATASET_NAME}.pkl")
if QUERY_ENTITY_CACHE_PATH.exists():
    with open(QUERY_ENTITY_CACHE_PATH, "rb") as f:
        query_entity_cache = pickle.load(f)
    print(f"[QueryEntityCache] Loaded {len(query_entity_cache)} cached query entities from {QUERY_ENTITY_CACHE_PATH}")
else:
    query_entity_cache = {}
    print(f"[QueryEntityCache] No cache found at {QUERY_ENTITY_CACHE_PATH}, starting empty")


def save_query_entity_cache() -> None:
    QUERY_ENTITY_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(QUERY_ENTITY_CACHE_PATH, "wb") as f:
        pickle.dump(query_entity_cache, f)

In [ ]:
retriever = QASARetriever(
    driver=driver,
    embedder=embedder,
    max_steps=3,
    decay=0.7,
    resource_threshold=0.01,
    seed_vector_fallback_k=5,
    top_k_entities=30,
    top_k_paths=30,
    query_entity_cache=query_entity_cache,
)

agent = AnswerAgent(retriever=retriever, llm_judge=True)

In [ ]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

GT_INDEX = 5
print(f"Loaded {len(ground_truth)} questions")
ground_truth[GT_INDEX]

## Single question demo

In [ ]:
gt = ground_truth[GT_INDEX]
print(f"Q: {gt['question']}")
print(f"A: {gt['answer']}")
print(f"Supporting: {gt['supporting_paragraphs']}\n")

result = agent.evaluate(gt["question"], gt["answer"])

print(f"\nPrediction: {result['prediction']}")
print(f"EM: {result['em']}, F1: {result['f1']:.3f}")
print(f"Seeds: {result['retrieval'].seed_entities}")
print(f"Extracted: {result['retrieval'].extracted_entities}")
print(f"Steps: {result['retrieval'].steps_taken}")
print(f"Paths: {len(result['retrieval'].paths)}")

In [ ]:
print("--- Knowledge graph paths (multi-hop chains) ---")
for p in result["retrieval"].paths:
    names, edges = p["names"], p["edges"]
    chain = names[0]
    for i, edge in enumerate(edges):
        rel = edge["relation"]
        if edge["forward"]:
            chain += f" --[{rel}]--> {names[i+1]}"
        else:
            chain += f" <--[{rel}]-- {names[i+1]}"
    print(f"  w={p['path_weight']:.4f}  {chain}")

print("\n--- Entity descriptions ---")
for e in result["retrieval"].entities:
    desc = e["description"]
    print(f"  {e['resource']:.4f}  {e['name']:30s}  {desc}")

print("\n--- Context sent to LLM ---")
print(result["paths_context"])
print(result["descriptions_context"])

## Batch evaluation

In [ ]:
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm import tqdm

N_EVAL = 1001
REPORT_EVERY = 25
WINDOW = 100
MAX_WORKERS = 20

assert isinstance(retriever, QASARetriever), (
    "Parallel evaluation requires the stateless, parallel-safe QASARetriever."
)

eval_results = [None] * N_EVAL
recent_em = deque(maxlen=WINDOW)
recent_f1 = deque(maxlen=WINDOW)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_to_idx = {
        pool.submit(agent.evaluate, gt["question"], gt["answer"]): i
        for i, gt in enumerate(ground_truth[:N_EVAL])
    }

    progress = tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Evaluating")
    for done_count, fut in enumerate(progress, start=1):
        idx = future_to_idx[fut]
        gt = ground_truth[idx]
        try:
            result = fut.result()
            eval_results[idx] = result
            recent_em.append(float(result["em"]))
            recent_f1.append(float(result["f1"]))
        except Exception as e:
            print(f"Error on {gt['question_id']}: {e}")

        if done_count % REPORT_EVERY == 0:
            valid = [r for r in eval_results if r is not None]
            n = max(1, len(valid))
            progress.set_postfix({
                "EM": f"{sum(r['em'] for r in valid) / n:.3f}",
                "F1": f"{sum(r['f1'] for r in valid) / n:.3f}",
                f"EM@{WINDOW}": f"{sum(recent_em) / max(1, len(recent_em)):.3f}",
                f"F1@{WINDOW}": f"{sum(recent_f1) / max(1, len(recent_f1)):.3f}",
                "ok": len(valid),
            })
            save_query_entity_cache()

save_query_entity_cache()
ok = sum(1 for r in eval_results if r is not None)
print(f"\n[QueryEntityCache] Saved {len(query_entity_cache)} cached query entities to {QUERY_ENTITY_CACHE_PATH}")
print(f"Completed {ok}/{N_EVAL} questions successfully")

In [ ]:
import pandas as pd

rows = [
    AnswerAgent.flatten_result(result, ground_truth[i]["question_id"])
    for i, result in enumerate(eval_results)
    if result is not None
]
df = pd.DataFrame(rows)
df.to_csv(f"eval-results-{DATASET_NAME}.csv", index=False)

# Print aggregates
print(f"Results on {len(df)} questions:")
print(f"  EM:  {df['em'].mean():.3f}")
print(f"  F1:  {df['f1'].mean():.3f}")
print(f"  LLM Accuracy:  {df['llm_accuracy'].mean():.3f}")
print(f"  Avg prompt tokens: {df['prompt_tokens'].mean():.0f}")
print(f"  Avg completion tokens: {df['completion_tokens'].mean():.0f}")
print(f"  Avg paths: {df['num_paths'].mean():.1f}")
print(f"  Avg entities: {df['num_entities'].mean():.1f}")

In [ ]:
driver.close()